# Notebook 03: Taxa + Taxa Clusters & Module Eigengenes + Singleton Genes 

Association testing between taxa + taxa cooperatives (clusters) and microbiome PCs vs. WGCNA module eigengenes + genes.

**Three comparisons:**
- **Part 5:** Taxa cooperatives vs module eigengenes + genes
- **Part 6:** Taxa cooperatives vs transcriptome PCs (gPCs)
- **Part 7:** mPCs vs module eigengenes + genes

**Input data:**
- **Microbiome side (Parts 8-9):** `cooperatives.csv` — CLR abundances for 125 taxa and cooperative members
- **Microbiome side (Part 10):** `taxa_abund_clr_corrected_in_20PC_coord.tsv` (15 PCs)
- **Part 8+10 transcriptome:** `*_expr_module_eigengenes.tsv` — WGCNA module eigengenes +
  individual probe expression (genes)
- **Part 9 transcriptome:** `*_PCs_after_correction_v2.txt` — transcriptome PCs

No inflation correction is applied in any part.

## Bootstrap: Environment Setup

In [ ]:
# --- Bootstrap ---
BASE_DIR <- normalizePath(file.path(getwd(), '..'))
DATA_DIR    <- file.path(BASE_DIR, 'Data')
CODE_DIR    <- file.path(BASE_DIR, 'code', 'CEDAR-master')
OUTPUT_DIR  <- file.path(BASE_DIR, 'notebooks', 'output')
PLOTS_DATA  <- file.path(BASE_DIR, 'code', 'data_for_plots')
TRANSCR_DIR       <- file.path(DATA_DIR, 'CEDAR1_transcriptome')
EXPR_PC_CORR_DIR  <- file.path(TRANSCR_DIR, 'Expr_data_Corr_for_PCs')
TRANSCR_PCS_DIR   <- file.path(TRANSCR_DIR, 'Momo-Dmitrieva2018')
MODULE_EIGEN_DIR  <- file.path(TRANSCR_DIR, 'Module_eigengenes_expr')
EIGEN_ANAL_DIR    <- file.path(TRANSCR_DIR, 'Expression_eignegenes_for_anal')
dir.create(OUTPUT_DIR, showWarnings = FALSE, recursive = TRUE)

suppressPackageStartupMessages({
    library(data.table)
    library(ade4)
    library(vegan)
    library(chemometrics)
    library(ggplot2)
    library(igraph)
    library(gplots)
    library(ape)
    library(stringr)
    library(jsonlite)
    library(lattice)
    library(tidyr)
    library(RhpcBLASctl)
    library(data.table)
    library(parallel)
    library(doParallel)
    library(foreach)
    library(dplyr)
    library(IRdisplay)
})
for (pkg in c('WGCNA', 'Rtsne', 'DirichletMultinomial', 'WebGestaltR', 'RCurl')) {
    if (requireNamespace(pkg, quietly = TRUE)) {
        suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    }
}

R_DIR <- file.path(BASE_DIR, 'notebooks', 'R')
source(file.path(R_DIR, 'functions_general.R'))
source(file.path(R_DIR, 'functions_figures.R'))
source(file.path(R_DIR, 'functions_association.R'))
source(file.path(R_DIR, 'functions_transcriptome.R'))

LOCATIONS   <- c('IL', 'TR', 'RE')
CELL_TYPES  <- c('CD4', 'CD8', 'CD14', 'CD15', 'CD19', 'PLA')
ALL_TYPES   <- c(LOCATIONS, CELL_TYPES)
N_TRANSCR_PCS_LOCATION <- c(IL = 59, TR = 50, RE = 53)
N_TRANSCR_PCS_CELL_TYPE <- c(CD4 = 38, CD8 = 35, CD14 = 36,
                          CD15 = 27, CD19 = 40, PLA = 23)
N_TRANSCR_PCS <- c(N_TRANSCR_PCS_LOCATION, N_TRANSCR_PCS_CELL_TYPE)

N_SIDAK_PERM = 1000

Loading required package: fitdistrplus

Loading required package: MASS


Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select


Loading required package: survival

Loading required package: tibble


Attaching package: ‘tibble’


The following object is masked from ‘package:igraph’:

    as_data_frame




In [2]:
# --- Setup Parallel Backend ---
num_cores <- 30

## Load and Prepare Input Data

In [3]:
# Output subdirectory
NB03_OUT <- file.path(OUTPUT_DIR, 'nb03')
dir.create(NB03_OUT, showWarnings = FALSE, recursive = TRUE)

In [4]:
# --- Taxa cooperatives (microbiome input for Parts 8 and 9) ---
# cooperatives.csv: 125 taxa x 315 samples (tab-separated despite .csv extension)
# Transpose to: samples x 125 taxa (required by corr.trPCs.vs.mPCs)
taxa.cooperat <- read.table.smart(file.path(DATA_DIR, 'cooperatives.csv'))
taxa.cooperat <- as.data.frame(t(taxa.cooperat))
cat('taxa.cooperat (transposed):', nrow(taxa.cooperat), 'samples x',
    ncol(taxa.cooperat), 'taxa\n')
cat('First 3 taxa:', paste(head(colnames(taxa.cooperat), 3), collapse = '\n  '), '\n')

# --- Microbiome PCs (microbiome input for Part 7) ---
microbiome.20PCs <- read.table.smart(file.path(DATA_DIR, 'taxa_abund_clr_corrected_in_20PC_coord.tsv'))
microbiome.15PCs <- microbiome.20PCs[, 1:15]

cat('mPCs for tissues and immune:', nrow(microbiome.15PCs), 'samples x', ncol(microbiome.15PCs), 'PCs\n')

taxa.cooperat (transposed): 315 samples x 125 taxa
First 3 taxa: k__Bacteria;p__Actinobacteria;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Bifidobacterium
  k__Bacteria;p__Actinobacteria;c__Coriobacteriia;o__Coriobacteriales;f__Coriobacteriaceae;g__Collinsella
  k__Bacteria;p__Bacteroidetes;c__Bacteroidia;o__Bacteroidales;f__Barnesiellaceae;g__Barnesiella 
mPCs for tissues and immune: 315 samples x 15 PCs


In [5]:
# --- Preparation: Read once ---
transcriptome_list <- list()
for(tn in ALL_TYPES){
    fname <- paste0(word(tn, 1, sep = '\\.'), '_PCs_after_correction_v2.txt')
    transcriptome_list[[tn]] <- read.table(file.path(TRANSCR_PCS_DIR, fname), row.names = 1, header = T)
}

In [6]:
# synonym variable
taxa_cooperatives = taxa.cooperat

# Load Module Eigengenes for Parts 5 & 7
# Files like: IL_expr_module_eigengenes.tsv
me_list <- list()
for(tn in c(LOCATIONS, CELL_TYPES)){
    fname <- paste0(word(tn, 1, sep = '\\.'), '_expr_module_eigengenes.tsv')
    me_list[[tn]] <- read.table(file.path(MODULE_EIGEN_DIR, fname), header = T, row.names = 1)
}

## Part 5: Taxa + Taxa Clusters vs Module Eigengenes + Genes

Test association between each of the taxa cooperatives + singleton taxa and each module eigengene + probe/gene in the expression data.

- Microbiome: 125 taxa cooperatives (CLR-corrected abundances)
- Transcriptome: module eigengene files containing probes + ME columns
- Method: `lm(expression ~ taxon)` via `anova()`
- Outlier removal: **neither** side (`remove.outliers = c(F, F)`)
- No inflation correction
- FDR applied separately per tissue/immune group

In [7]:
# --- Part 5a: Tissues ---
prepared_5a <- prepare_data_universal(
    microb_df = taxa_cooperatives,
    transcr_list = me_list[LOCATIONS],
    exclude_mode = 'first_1_column',
    outlier_modes = c(FALSE, FALSE)
)

In [8]:
# 40 sec
pval_real_5a <- run_universal_association(
	prepared_5a,
	m_prefix="",
	t_prefix="",
	inflation_correct = FALSE,
	QQplot.file = file.path(NB03_OUT, 'QQ_Part5a_TaxaClust_MEs_3locs.pdf'),
    main = 'Taxa + taxa clusters vs module eigengenes + genes (3 tissues)'	
)

In [9]:
# 5 min
res_5a <- run_neff_pipeline(
    pval_observed = pval_real_5a,
    prepared_data = prepared_5a,
    microb_base = taxa_cooperatives,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = NULL # no inflation correction
)


Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0299 
Neff Estimation Complete (354s)
Total Tests: 1335500 
Effective Independent Tests (Neff): 1059338.2 
Reduction in penalty: 20.7 %

p Sidak <= 0.15: 1 
TR.k__Bacteria;p__Firmicutes;c__Erysipelotrichia;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Catenibacterium.X3830671 
                                                                                                               0.02903148 

p Hochberg <= 0.15: 1 
TR.k__Bacteria;p__Firmicutes;c__Erysipelotrichia;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Catenibacterium.X3830671 
                                                                                                               0.03714155 


In [10]:
write.pvalue.vectors.to.file(
    file.path(NB03_OUT, "pvals_Part5a_TaxaClust_vs_MEs_3locations.tsv"),    
    Original = pval_real_5a,
    Hochberg = res_5a$pval_hochberg,
    Sidak = res_5a$pval_sidak,     
    signif_ = 4
)

In [11]:
# --- Part 5b: Immune Cells ---
prepared_5b <- prepare_data_universal(
    microb_df = taxa_cooperatives,
    transcr_list = me_list[CELL_TYPES],
    exclude_mode = 'first_1_column',
    outlier_modes = c(FALSE, FALSE)
)

In [12]:
# 1 min
pval_real_5b <- run_universal_association(
	prepared_5b,
	m_prefix="",
	t_prefix="",
	inflation_correct = FALSE,
	QQplot.file = file.path(NB03_OUT, 'QQ_Part5b_TaxaClust_MEs_6cells.pdf'),
    main = 'Taxa + taxa clusters vs module eigengenes + genes (6 immune cell types)'	
)

In [13]:
# 8-9 min
res_5b <- run_neff_pipeline(
    pval_observed = pval_real_5b,
    prepared_data = prepared_5b,
    microb_base = taxa_cooperatives,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = NULL # no inflation correction
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0226 
Neff Estimation Complete (530.6s)
Total Tests: 1891125 
Effective Independent Tests (Neff): 1859324.03 
Reduction in penalty: 1.7 %

p Sidak <= 0.15: 7 
           CD4.k__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhodospirillales;f__uncultured;g__Azospirillum.sp..47_25.X4810292 
                                                                                                                              0.08202581 
                 CD4.k__Bacteria;p__Firmicutes;c__Erysipelotrichia;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Turicibacter.X5900471 
                                                                                                                              0.07649648 
               CD8.k__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhodospirillales;f__uncultured;g__Azospirillum.sp..47_25.ME13 
                                                      

In [14]:
write.pvalue.vectors.to.file(
    file.path(NB03_OUT, "pvals_Part5b_TaxaClust_vs_MEs_6cells.tsv"),
    Original = pval_real_5b,
    Hochberg = res_5b$pval_hochberg,
    Sidak = res_5b$pval_sidak, 
    signif_ = 4
)

## Part 6: Taxa + Taxa Clusters vs Transcriptome PCs 

Test association between each of the 125 taxa + taxa cooperatives and transcriptome PCs.

- Microbiome: 125 taxa + taxa cooperatives (same as Part 8)
- Transcriptome: PCs from `Momo-Dmitrieva2018/`
- Method: `lm(gPC ~ taxon)` via `anova()`
- Outlier removal: **only transcriptome** side (`remove.outliers = c(F, T)`), IQR × 5
- No inflation correction

In [15]:
# --- Part 6a: Tissues ---
prepared_6a <- prepare_data_universal(
    microb_df = taxa_cooperatives,
    transcr_list = transcriptome_list[LOCATIONS],
    n_pcs = N_TRANSCR_PCS_LOCATION,
    exclude_mode = 'last_columns',
    outlier_modes = c(FALSE, TRUE), 
    iqr_mult = 5
)

In [16]:
# 1 sec
pval_real_6a <- run_universal_association(
	prepared_6a,
	m_prefix="",
	t_prefix="gPC",
	inflation_correct = FALSE,
	QQplot.file = file.path(NB03_OUT, 'QQ_Part6a_TaxaClust_gPCs_3locs.jpg'),
    main = 'Taxa + taxa cooperatives vs gene expression PCs (3 locations)'
)

In [17]:
# 
res_6a <- run_neff_pipeline(
    pval_observed = pval_real_6a,
    prepared_data = prepared_6a,
    microb_base = taxa_cooperatives,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = NULL # no inflation correction
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0158 
Neff Estimation Complete (7.2s)
Total Tests: 20250 
Effective Independent Tests (Neff): 19834.67 
Reduction in penalty: 2.1 %

p Sidak <= 0.15: 1 
IL.k__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Lachnospiraceae;g__Lachnospiraceae.UCG.008.gPC34 
                                                                                                      0.1351312 

p Hochberg <= 0.15: 1 
IL.k__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Lachnospiraceae;g__Lachnospiraceae.UCG.008.gPC34 
                                                                                                      0.1482169 


In [18]:
write.pvalue.vectors.to.file(
    file.path(NB03_OUT, "pvals_Part6a_TaxaClust_gPCs_3locs.tsv"),
    Original = pval_real_6a,
    Hochberg = res_6a$pval_hochberg,
    Sidak = res_6a$pval_sidak,    
    signif_ = 4
)

In [19]:
# --- Part 6b: Immune Cells ---
prepared_6b <- prepare_data_universal(
    microb_df = taxa_cooperatives,
    transcr_list = transcriptome_list[CELL_TYPES],
    n_pcs = N_TRANSCR_PCS_CELL_TYPE,
    exclude_mode = 'last_columns',
    outlier_modes = c(FALSE, TRUE),
    iqr_mult = 5
)

In [20]:
pval_real_6b <- run_universal_association(
    prepared_6b,
    m_prefix="",
    t_prefix="gPC",
    inflation_correct = FALSE,
    QQplot.file = file.path(NB03_OUT, 'QQ_Part6b_TaxaClust_gPCs_6cells.jpg'),
    main = 'Taxa + taxa cooperatives vs gene expression PCs (6 immune cell types)'
)

In [21]:
res_6b <- run_neff_pipeline(
    pval_observed = pval_real_6b,
    prepared_data = prepared_6b,
    microb_base = taxa_cooperatives,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = NULL # no inflation correction
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0139 
Neff Estimation Complete (10.2s)
Total Tests: 24875 
Effective Independent Tests (Neff): 26141.69 
Reduction in penalty: -5.1 %

p Sidak <= 0.15: 2 
CD8.k__Bacteria;p__Firmicutes;c__Erysipelotrichia;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Catenibacterium.gPC5 
                                                                                                           0.002600898 
       CD19.k__Bacteria;p__Firmicutes;c__Erysipelotrichia;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Dielma.gPC12 
                                                                                                           0.079077822 

p Hochberg <= 0.15: 2 
CD8.k__Bacteria;p__Firmicutes;c__Erysipelotrichia;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Catenibacterium.gPC5 
                                                                                                           0.002478097 
    

In [22]:
write.pvalue.vectors.to.file(
    file.path(NB03_OUT, "pvals_Part6b_TaxaClust_gPCs_3locs.tsv"),
    Original = pval_real_6b,
    Hochberg = res_6b$pval_hochberg,
    Sidak = res_6b$pval_sidak,    
    signif_ = 4
)

## Part 7: mPCs vs Module Eigengenes + genes

Tests pairwise associations between microbiome principal components and WGCNA
module eigengenes (+ individual genes/probes).

- **Microbiome features:** 15 mPCs
- **Transcriptome features:** All columns of `*_expr_module_eigengenes.tsv`
  (individual probes + WGCNA module eigengenes per tissue/cell type)
- **Test:** `lm(eigengene ~ mPC)` via `anova()`
- **Outlier removal:** neither side (`remove.outliers = c(F, F)`), IQR threshold = 5

In [23]:
# --- Part 7a: Tissues ---
prepared_7a <- prepare_data_universal(
    microb_df = microbiome.15PCs,
    transcr_list = me_list[LOCATIONS],
    exclude_mode = 'first_1_column',
    outlier_modes = c(FALSE, FALSE),
    iqr_mult = 5
)

In [24]:
# 
pval_real_7a <- run_universal_association(
	prepared_7a,
	m_prefix="mPC",
	t_prefix="",
	inflation_correct = FALSE,
	QQplot.file = file.path(NB03_OUT, 'QQ_Part7a_mPCs_vs_MEs_3locs.jpg'),
    main = 'Taxonomic PCs vs module eigengenes + genes (3 locations)'
)

In [25]:
res_7a <- run_neff_pipeline(
    pval_observed = pval_real_7a,
    prepared_data = prepared_7a,
    microb_base = microbiome.15PCs,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = NULL # no inflation correction
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0128 
Neff Estimation Complete (46.6s)
Total Tests: 160260 
Effective Independent Tests (Neff): 128062.57 
Reduction in penalty: 20.1 %

p Sidak <= 0.15: 4 
 TR.mPC1.X3370288  TR.mPC1.X3780575 TR.mPC11.X3830671  TR.mPC1.X1030286 
      0.051607674       0.012517854       0.035851538       0.009018355 

p Hochberg <= 0.15: 4 
 TR.mPC1.X3370288  TR.mPC1.X3780575 TR.mPC11.X3830671  TR.mPC1.X1030286 
       0.06630773        0.01576386        0.04568874        0.01133694 


In [26]:
write.pvalue.vectors.to.file(
    file.path(NB03_OUT, "pvals_Part7a_mPCs_vs_MEs_3locs.tsv"),
    Original = pval_real_7a,
    Hochberg = res_7a$pval_hochberg,
    Sidak = res_7a$pval_sidak,    
    signif_ = 4
)

In [27]:
# --- Part 7b: Immune cells ---
prepared_7b <- prepare_data_universal(
    microb_df = microbiome.15PCs,
    transcr_list = me_list[CELL_TYPES],
    exclude_mode = 'first_1_column',
    outlier_modes = c(FALSE, FALSE),
    iqr_mult = 5
)

In [28]:
# 
pval_real_7b <- run_universal_association(
	prepared_7b,
	m_prefix="mPC",
	t_prefix="",
	inflation_correct = FALSE,
	QQplot.file = file.path(NB03_OUT, 'QQ_Part7b_mPCs_vs_MEs_6cells.jpg'),
    main = 'Taxonomic PCs vs module eigengenes + genes (6 immune cell types)'
)

In [29]:
res_7b <- run_neff_pipeline(
    pval_observed = pval_real_7b,
    prepared_data = prepared_7b,
    microb_base = microbiome.15PCs,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = NULL # no inflation correction
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0236 
Neff Estimation Complete (69s)
Total Tests: 226935 
Effective Independent Tests (Neff): 201362.32 
Reduction in penalty: 11.3 %

p Sidak <= 0.15: 2 
CD4.mPC11.X4810292     CD8.mPC11.ME13 
        0.02230105         0.03676340 

p Hochberg <= 0.15: 2 
CD4.mPC11.X4810292     CD8.mPC11.ME13 
        0.02541773         0.04221289 


In [30]:
write.pvalue.vectors.to.file(
    file.path(NB03_OUT, "pvals_Part7b_mPCs_vs_MEs_6cells.tsv"),
    Original = pval_real_7b,
    Hochberg = res_7b$pval_hochberg,
    Sidak = res_7b$pval_sidak,    
    signif_ = 4
)